Exploring the CF Trust registry data from 2008 to 2018 data to evaluate if model output (synthesizing FEV1, FEF25-75 on 2 days) can be used to improve ML achieved from each indidivually 

In [3]:
import logging

import pandas as pd

import data.helpers as dh
import models.var_builders as var_builders
import data.cfr_data_19_23 as cfrd
import data.breathe_data as bd

In [17]:
df = pd.read_csv(
    dh.get_path_to_main() + f"DataFiles/CFR/CF2015.csv",
    # usecols=['ID'],
    encoding='cp1252' 
)
# 2008 hasn't FEF25-75

/var/folders/zq/v2r6yn111s3gpdf8lzf72xvw0000gn/T/ipykernel_45181/1905519900.py:1: DtypeWarning: Columns (56,59,60,69,79,80,81,87,90,91,93,94,95,96,97,99,100,101,104,113,117,118,119,124,125,131,132,133,137,142,143,148,170,193,198,217,231,232,234,243,247,248,269,279) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(


In [30]:
import plotly.express as px

col = 'Intravenous (IV) Antibiotic Overall Days '
fig = px.histogram(
    df,
    x=col,
    nbins=50,
    histnorm="percent",
    # nbins=30,
    text_auto=".2f",
    title="Histogram of IV Antibiotic Overall Days",
    labels={col: "Number of IV Antibiotic Overall Days", "count": "Number of Patients"}
)
fig.update_traces(textangle=270)
fig.show()



In [33]:
import plotly.express as px
import numpy as np

col = 'Intravenous (IV) Antibiotic Overall Days '

# Create bin edges: 0 (single bin), then 1-10, 11-20, ..., max value present
max_val = df[col].max(skipna=True)
upper = np.ceil(max_val / 10) * 10 if not pd.isnull(max_val) else 100
bins = [0, 1] + list(range(11, int(upper)+10, 10))
# 0 | 1-10 | 11-20 | ... (1-10, 11-20, ..., last bin includes upper outliers)

def bin_label(val):
    if val == 0:
        return "0"
    elif 1 <= val <= 10:
        return "1-10"
    else:
        lower = ((val-1)//10)*10 + 1
        upper = lower + 9
        return f"{lower}-{upper}"

df['_iv_abx_bins'] = df[col].apply(bin_label)

fig = px.histogram(
    df,
    x='_iv_abx_bins',
    histnorm="percent",
    category_orders={'_iv_abx_bins': ["0", "1-10"] + [f"{i}-{i+9}" for i in range(11, int(upper), 10)]},
    text_auto=".2f",
    title="Histogram of IV Antibiotic Overall Days (Binned)",
    labels={'_iv_abx_bins': "Number of IV Antibiotic Overall Days (binned)", "count": "Number of Patients"}
)
fig.update_traces(textangle=90)
fig.update_xaxes(type='category')
fig.show()



In [45]:
df.shape

(9587, 283)

In [43]:
# for year in [2008, 2009, 2010, 2011, 2012, 2013, 2014]:
for year in [2016, 2017, 2018]:
    print(year)
    df1 = pd.read_csv(
        dh.get_path_to_main() + f"DataFiles/CFR/CF{year}.csv",
        # usecols=['ID'],
        encoding='cp1252' 
    )
    print(df1.columns.to_list())

2016


/var/folders/zq/v2r6yn111s3gpdf8lzf72xvw0000gn/T/ipykernel_45181/365697482.py:4: DtypeWarning:

Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.

/var/folders/zq/v2r6yn111s3gpdf8lzf72xvw0000gn/T/ipykernel_45181/365697482.py:4: DtypeWarning:

Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.



['ID', 'Gender', 'Genetic Mutation Allele 1', 'Genetic Mutation Allele 2', 's02genotypemut1specify', 's02genotypemut2specify', 'Height', 'Centile Height', 'Weight', 'Centile Weight', 'BMI', 's01encounterageyears', 's01encounteragemonths', 's03deathhaspatientdied', 's03deathcause', 'death_dt_anon', 'received_trans', 's10transplantdate', 'trans_lung_bilat', 'trans_lung_heart', 'trans_lung_lb_cadaveric', 's10transplantevaluation', 's10transplantevaluationoutcome', 's10transplantevaluationdate', 'FEV1', 'FEV1 Predicted', 'Best FEV1', 'Best FEV1 Predicted', 'smokeCigarettes', 'cmpl_diab_hgba1c_val', 'cmpl_hemoptysis', 'HemoptysisScanty', 'HemoptysisSevere', 'HemoptysisMassive', 'HemoptysisModerate', 'cmpl_CFRD', 'cmpl_diab_with_hyperglycemia', 'cmpl_other_glucose_ab', 'cfrd_treatment', 'cmpl_diab_gluc_intoler', 'cmpl_diab_dietary_chg', 'cmpl_oral_hypoglycemic_agnts', 'cmpl_diab_inter_insul', 'cmpl_microalbuminuria', 'cmpl_diabetes', 'cmpl_retinopathy', 'cmpl_cancer', 'cmpl_Arrhythmia', 'cmp

/var/folders/zq/v2r6yn111s3gpdf8lzf72xvw0000gn/T/ipykernel_45181/365697482.py:4: DtypeWarning:

Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.

